# GraphRAG: AI Copyright & Governance Knowledge Graph. 
This notebook builds a full **GraphRAG pipeline** on a pre-scraped dataset of news articles,  
about AI copyright, governance, and intellectual property.
### Pipeline overview
| Step | What happens |
|---|---|
| 1 | Load and prepare your scraped article dataset.|
| 2 | Define a domain ontology for AI copyright/governance.|
| 3 | Extract entities + relationships WITH descriptions using a custom extractor.|
| 4 | Build a knowledge graph with community detection baked in.|
| 5 | Generate LLM community summaries.|
| 6 | Visualize the graph interactively.|
| 7 | Query using GraphRAG |

In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
import networkx as nx 
from dotenv import load_dotenv 
from llama_index.core import Document, PropertyGraphIndex, Settings
from llama_index.core.graph_stores.types import (
    EntityNode, KG_NODES_KEY , KG_RELATIONS_KEY , Relation
)
from llama_index.core.graph_stores import SimplePropertyGraphStore 
from llama_index.core.llms.llm import LLM 
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import TransformComponent, BaseNode 
from llama_index.core.async_utils import run_jobs 
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.llms.openai import OpenAI
from llama_index.llms.google_genai import GoogleGenAI
from graspologic.partition import hierarchical_leiden
# Patch the event loop so LlamaIndex async calls work inside Jupyter
nest_asyncio.apply() 
load_dotenv() 
print("Import Ready")

/Users/mrvishal/Desktop/mangal_world/AI_agents/projects/GraphAPI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import Ready


In [46]:
g_llm = GoogleGenAI(model="gemini-3.5-flash")   # use the name that worked above
print(g_llm.complete("Say Hello"))

Hello! How can I help you today?


In [ ]:
# LLM : LLM = OpenAI(model = "gpt-4o-mini")
# -- Pipeline Parameters --------------------------------------------
MAX_ARTICLES = 50
MAX_PATH_PER_CHUNK = 20 
NUM_WORKERS = 4
MAX_CLUSTER_SIZE = 10 
GRAPH_OUTPUT_FILE = "ai_copyright_graph.html" 
Settings.llm= g_llm
print(f"Use {Settings.llm.model} in this directory.")

Use gemini-3.5-flash in this directory.


## 3. Define the Ontology 
The ontology is the schema of our Knowledge Graph. It controls exactly what kinds of entities and relationships the LLM is allowed to extract.  
### Why it matters
Without a defined schema, the LLM will:
* Invest differnt entity types per document ("AI_COMPANY" vs "TECH_FIRM" vs "ORGANIZATION").
* Use inconsistent relationship labels for the same concepts.
* Create duplicate node for the same entity under slightly different names.

The ontology constrains extraction to a fixed vocabulary, so the graph stays consistent across all documents. 
### For this particular use case - AI copyright & governance.
I defined below 7 entity types, 8 relationship type.  
But this really depends on your use case and what you want to research.

In [2]:
# -- Entity types ------------------
# We'll embed these directly into the extraction prompt below.

ENTITY_TYPES = [
    "ORGANISATION",     # Companies, labs, industry groups (OpenAI, Google, RIAA) 
    "PERSON",           # Executives, policymakers, judges (Sam Altman, Thierry Breton) 
    "LEGISLATION",      # Laws, acts, regulations (EU AI Act, DMCA, Copyright Act) 
    "LEGAL_CASE",       # Lawsuits, court rulings (NYT v. OpenAI, Getty v. Stability AI) 
    "CONCEPT",          # Abstract ideas: fair use, training data, IP rights
    "GOVERNMENT",       # Nations, regulatory bodies, courts (EU, US Copyright Office) 
    "AI_SYSTEM"         # Specific models or products (GPT-4, Stable Diffusion , Gemini) 
]

# --  Relationship types ---------------
RELATION_TYPES = [
    "FILED_AGAINST",    # Plaintiff Organisation/Person -> Legal_Case 
    "DEFENDANT_IN",     # Organization/Person -> Legal_Case 
    "REGULATES",        # GOVERNMENT/LEGISLATION  -> Organization / AI_SYSTEM 
    "ADVOCATES_FOR",    # ORGANISATION/PERSON  -> CONCEPT or policy position
    "TARINED_ON",       # AI_SYSTEM -> Concept or dataset type
    "PART_OF",          # PERSON -> ORGANIZATION
    "REFRENCES",        # LEGAL_CASE /LEGISLATION -> CONCEPT 
    "OPPOSES"           # ORGANIZATION/GOVERNMENT -> LEGISLATION/CONCEPT
]
print("✅ Ontology:") 
print(f" Entity types: {ENTITY_TYPES}")
print(f" Relationship types: {RELATION_TYPES}")

✅ Ontology:
 Entity types: ['ORGANISATION', 'PERSON', 'LEGISLATION', 'LEGAL_CASE', 'CONCEPT', 'GOVERNMENT', 'AI_SYSTEM']
 Relationship types: ['FILED_AGAINST', 'DEFENDANT_IN', 'REGULATES', 'ADVOCATES_FOR', 'TARINED_ON', 'PART_OF', 'REFRENCES', 'OPPOSES']


## 4. Extraction Prompt 
This is where our ontology gets embedded into the LLM's instructions.
Most GraphRAG implementations extract bare triplets: (OpenAI, DEFENDANT_IN, NYT v. OpenAI).
This prompt extracts **desciptions alongside every entity and relatioship**:
* Entity desciption : "OpenAI is an AI research company that developed GPT-4, currently facing multiple copyright lawsuits"
* Relationship description : "OpenAI is named as defendant in the New York Times lawsuit over  allegedly using copyrighted articles as training data.


These descriptions flow through the community summaries, making them far richer and enabling much better final answer.

In [5]:
# Build the entity and relationship type strings dynamically from our ontology.
entity_types_str = ", ".join(ENTITY_TYPES) 
relation_types_str = ", ".join(RELATION_TYPES) 
print(entity_types_str)

KG_TRIPLET_EXTRACT_TMPL = f"""
                        -- GOAL -- 
                        Given a news article about AI copyright, governance, or intellectual property.
                        identify all entities mentioned in the article and their relationships.
                        Extract up to {{max_knowlege_triplets}} entity-relation triplets.

                        - Allowed Entity Types -
                        {entity_types_str} 

                        - Allowed Relationship Types - 
                        {relation_types_str} 

                        - Steps - 
                        1. Identify ALL entities. For each entity extract: 
                            - name : Name of the entity, capitalized
                            - type : One of the allowed entity types above
                            - description : A  brief description of the entity and its role in AI copyright/governance 
                        2. Identify relationships between entities. For each extract:
                            - source : name of the source entity
                            - target : name of the target entity 
                            - relation : one of the allowed relationship types above
                            - description : a sentence explaining why and how these entities are related.        
                        - Real Data -
                        ###################
                        text:{{text}}
                        ###################
                            
                        """
print("✅ Extraction prompt ready") 
print(f"\nPreview (first 300 chars):\n {KG_TRIPLET_EXTRACT_TMPL[:300]} . . .")


ORGANISATION, PERSON, LEGISLATION, LEGAL_CASE, CONCEPT, GOVERNMENT, AI_SYSTEM
✅ Extraction prompt ready

Preview (first 300 chars):
 
                        -- GOAL -- 
                        Given a news article about AI copyright, governance, or intellectual property.
                        identify all entities mentioned in the article and their relationships.
                        Extract up to {max_knowlege_triplets} en . . .


### Pydantic Extraction Model.

Instead of parsing the raw llm output using regex, we define Pydantic models that describe exactly what we want the LLM to return. LlamaIndex passes to OpenAI as a function schema, so the response is structured JSON -- validated and typed automatically.
Three model are defined:
* **ExtractedEntity** -- a single enity name , type and description
* **ExtractionRelationship** -- a relationship between two entities: source, target, relation, description.
* **ExtractionResult** -- the top-level wrapper containing a list of each.  


EntityTypeStr and RelationTypeStr are Literal types built from the ontlogy defined in the previous cell. This means the LLM cannot return an invalid type -- Pydantic will reject it before it ever reaches our code.  

In [28]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, List

EntityTypeStr = Literal[tuple(ENTITY_TYPES)] 
RelationTypeStr = Literal[tuple(RELATION_TYPES)] 

class ExtractedEntity(BaseModel):
    name : str = Field(description = "Name of the entity, capitalized") 
    type : EntityTypeStr = Field(description = "One of the allowed entity types") 
    description : str = Field(description = "Brief description of the entity and its role.") 

class ExtractedRelationship(BaseModel):
    source : str = Field(description = "Name of the source entity.") 
    target : str = Field(description= "Name of the target entity.")
    relation : RelationTypeStr = Field(description = "One of the allowed relationship types") 
    description : str = Field(description= "Sentence explaining the relationship") 

class ExtractionResult(BaseModel):
    entities : List[ExtractedEntity] = Field(default_factory=list)
    relationships : List[ExtractedRelationship] = Field(default_factory=list)

print("✅ Pydantic extraction model defined.")


✅ Pydantic extraction model defined.


## GraphRAG Extractor
This is the core extraction component. It sends each text chunk to the LLM with out ontology constrained prompt, parses the response, and stores the extracted entities and relationships as structured objects on each node. 

### Why build a custom extractor ?
LlamaIndex's built-in SchemaLLMPathExtractor extracts enitity/relationship labels but drops descriptions. By building our own extractor : 
* Every EntityNode carries a entity_desciption property 
* Every Relation carries a relationship_description property 
* These flow through to community summaries, making them far richer

These extractor runs asynchronously with nums_worker = 4 - processing 4 chunks in parallel, which speed up the extraction significantly without increasing API cost.

In [52]:
class GraphRAGExtractor(TransformComponent):
    # TransformComponent is mandatary for the nodes in and nodes out. (node : splitted chunks).
    """
    Extracts entities and relatonships WITH descriptions from each text chunk.
    Uses Pydantic structured output (via OpenAI fucntion calling) to guarantee that every entity has a valid type and every relationship uses a allowed label.

    Data flow for a single chunk :
    Raw text 
        -> LLM (astructured_predict) 
        -> ExtractionResult(
                            entities = [
                            ExtractedEntity(name="OpenAI" , type = "ORGANIZATION" , description = "AI lab...") 
                            ExtractedEntity(name="NYT v. OpenAI" , type = "LEGAL_CASE" , description = "Copyright lawsuit...") 
                            ],
                            relationships = [
                            ExtractedRelationship(source="OpenAI", target="NYT v. OpenAI",
                                        relation="DEFENDANT_IN", description="OpenAI is named defendant..."),
                            ]
                            )
        → EntityNode(name="OpenAI", label="ORGANIZATION", properties={...})
        → EntityNode(name="NYT v. OpenAI", label="LEGAL_CASE", properties={...})
        → Relation(label="DEFENDANT_IN", source_id=<OpenAI id>, target_id=<NYT v. OpenAI id>)
    """


    llm: LLM = Field(default_factory=lambda:Settings.llm) 
    extract_prompt : PromptTemplate = Field(default_factory= lambda: PromptTemplate(KG_TRIPLET_EXTRACT_TMPL)) 
    num_workers : int = 4 
    max_paths_per_chunk : int = 10   

    @field_validator("extract_prompt" , mode = "before") 
    @classmethod
    def coerce_to_prompt_template(cls,v):
        print(f"Validator class | cls = {cls.__name__} | v = {str(v)[:100]}") 
        return PromptTemplate(v) if isinstance(v,str) else v 
    
    def __call__(self , nodes , show_progress = False , **kwargs) :
        """
        Entry point called by LlamaIndex when this extractor runs in a pipeline.
            Takes the text chunks (nodes), extracts entities and relationships from each one with the LLM, and returns the same nodes with the results
            stored in their metadata.
        Args:
            nodes: List of BaseNode objects (text chunks from the splitter).
            show_progress: If True, shows a progress bar.
            **kwargs: Extra options passed through to acall.
        Returns:
        List of nodes with extracted entities and relationships in metadata.
        """
        print(f"__call__ | {type(self).__name__} | nodes = {len(nodes)} | first = {nodes[0].text[:100]!r}")
        return asyncio.run(self.acall(nodes , show_progress=show_progress , **kwargs))
    
    async def _aextract(self, node:BaseNode) -> BaseNode :
        text = node.get_content(metadata_mode="llm") # # Returns the chunk text prefixed with its metadata (excluding keys hidden from the LLM), as the LLM should see it.
        ''' 
        The four modes (the MetadataMode options)
        Mode   |  What it returns
        "none" :	Only the text, no metadata
        "all"  :	Text plus all metadata
        "llm"  :	Text plus metadata, except keys in excluded_llm_metadata_keys
        "embed":	Text plus metadata, except keys in excluded_embed_metadata_keys
        '''

        # ---- STEP - 1 Call the LLM ------------------------------------------------
        # astructured_predict uses OpenAI / gemini functon calling to return validated ExtractionResult format
        # so entity types and relation labels guaranteed valid. 
        try:
            result = await self.llm.astructured_predict(
                ExtractionResult,  # Pydantic schema the LLM output must follow
                self.extract_prompt, # prompt template with the extraction instructions
                text=text,    # chunk text filled into the template's {text} slot
                max_knowledge_triplets=self.max_paths_per_chunk, # # cap on triplets per chunk, filled into {max_knowledge_triplets}
            ) 
            entities = result.entities 
            relationships = result.relationsips 
        except Exception as e:
            print(f"Extraction error: {str(e)}")
            entities, relationships = [], []
        
        # For the first the KG_NODES_KEY and KG_RELATIONS_KEY is not avialable so its take empty list [] 
        existing_nodes = node.metadata.pop(KG_NODES_KEY , []) # node = Document(text , metadata)
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY , []) 
        base_metadata = node.metadata.copy()
        # ----- STEP - 2 Entities -> EntityNode objects ------------------------------- 
        # e.g. ExtractedEntity(name="OpenAI" , type = "ORGANIZATION" , description="...") 
        # becomes EntityNode(name = "OpenAI" , label = "ORGANIZATIOn" ,  properties = {"..."}) 
            
        existing_nodes += [
            EntityNode(
                name = entity.name,
                label = entity.type,
                properties = {**base_metadata , "entity_description" : entity.description},
            )
            for entity in entities
        ]
    
        # ----- STEP - 3 Relationships -> Relation objects ------------------------------- 
        # entity_lookup lets us attach the correct label to relationship source and target nodes.
        # e.g. {"OpenAI" : "ORGANISATION" , "NYT v. OpenAI" : "LEGAL_CASE"} 
        entity_lookup = {e.name : e.type for e in entities} 
        print("[GraphRAGExtractor][ENTITY_LOOKUP] : ",entity_lookup)
        for rel in relationships: # name in entity is source in relation

        # Build a source and a target EntityNode for this relationship.
        # Their ids equal their names, So they point at the same entities.
            source_node = EntityNode(
                name = rel.source ,
                label = entity_lookup.get(rel.source, "ENTITY"), # type
                properties = base_metadata
            )
            target_node = EntityNode(
                name = rel.target,
                label = entity_lookup.get(rel.target, "ENTITY"),
                properties = base_metadata,
            )
            # Only add endpoint nodes not already captured from the entity pass.
            # Safety net: only append an endpoint if step 2 did not already create it.
            # In the normal case both names are in entity_lookup, so nothing is added
            # and the relation simply links to the existing entities by id.
            if rel.source not in entity_lookup:
                existing_nodes.append(source_node)
            if rel.target not in entity_lookup:
                existing_nodes.append(target_node) 
            
            existing_relations.append(Relation(
                label = rel.relation,
                source_id = source_node.id ,
                target_id = target_node.id ,
                properties = { **base_metadata , "relationship_description" : rel.description }
            ))

        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations

        return node
    

    async def acall(self,nodes, show_progress= False, **kwargs):
        """Process all the workers parallel using num_workers async workers.""" 
        jobs= [self._aextract(node) for node in nodes] # this jobs holds the coroutine objects. 
        return await run_jobs(
            jobs,
            workers = self.num_workers,
            show_progress = show_progress ,
            desc = "Extracting triplets" 
        ) 
    

'OpenAI'

'OpenAI'

In [ ]:
# Wrap each article as a LlamaIndex Document
# full_text is the main content; everything else is metadata
nodes = [
    Document(
        text=str(row["full_text"]),
        metadata={
            "title":  str(row.get("title",  "")),
            "source": str(row.get("source", "")),
            "date":   str(row.get("date",   "")),
        }
    )
    for _, row in df.iterrows()
]

print(f"✅ Created {len(nodes)} LlamaIndex documents, stored as nodes") 



## Build the Knowledge Graph 
Now we wire everything and run the extraction pipeling.
PropertyGraphIndex handles the full workflow:
1. Passes each chunk to GraphRAGExtractor.
2. The extractor calls the LLM with our ontology constrained prompt.
3. Parsed entities and relationship stored in GraphRAGStore

-> This is most time-consuming steps! 



In [39]:
# Instantiate the extractor with our ontology prompt.
kg_extractor = GraphRAGExtractor(
    llm = Settings.llm,
    extract_prompt = KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk = MAX_PATH_PER_CHUNK,
    num_workers = NUM_WORKERS 
)
result = kg_extractor
# GRAPHRAGStore is both the graph database and the community detection engine.
# graph_store = GraphRAGStore() 
# print("Building knowledge graph... (this may take a few minutes.)")

Validator class | cls = GraphRAGExtractor | v = 
                        -- GOAL -- 
                        Given a news article about AI copyright
